# Lab 9: Classification Evaluation

**Business question:** Which classifier errors are acceptable for an urgent-complaint queue?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | Saved Module 8 predictions for the customer messages | about 30 min | Run outputs, saved files and the checkpoint |
| 2 Practice | CFPB complaint extract; saved Module 8 predictions | about 75–90 min (Exercise 2 means reading ten complaints) | Three exercises and the interpretation |
| 3 Challenge | CFPB complaints | optional (includes calibration) | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

Keep the **Lab Companion** for Module 9 open beside this notebook. It explains each step and shows the output you should expect.

A classifier gives a **probability**. A business needs a **decision**: act now, ask a person, or wait. In this lab you turn probabilities into a decision policy. You choose it on the **validation** split with a cost table. Then you **freeze** it and check it **once** on the sealed **test** split.

**Important:** do not use Run All for Part 2 until you have finished Exercise 3. The test split opens only after you have written your policy and set `READY_TO_FREEZE = True`. Until then, the test cells print a message and keep the test split sealed.

## Part 1 Guided core

### 1.1 Load the libraries

Select the `ban6403` kernel, then run the cell below. It loads the libraries we use in this lab. `datetime` gives the time stamp when you freeze your policy in 2.9.

In [ ]:
from pathlib import Path                  # file and folder paths
from datetime import datetime, timezone   # time stamps for the frozen policy
import json                               # writes and reads the frozen policy file
import numpy as np                        # arrays of numbers
import pandas as pd                       # tables
import matplotlib.pyplot as plt           # the precision-recall plot
from sklearn.feature_extraction.text import TfidfVectorizer  # the Module 4 word features
from sklearn.linear_model import LogisticRegression          # the classifier
from sklearn.pipeline import make_pipeline                   # chains TF-IDF and the classifier
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve  # the metrics

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the data is one folder up, in `../data`. Your own results go to `../outputs/m09`, including the frozen policy in 2.9.

In [ ]:
# Folder paths, relative to the notebooks folder.
DATA = Path("../data")
OUTPUT_DIR = Path("../outputs/m09")
# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 Load the saved Module 8 predictions

In Module 8 you fitted a TF-IDF logistic regression on the 13 development messages and predicted the 10 protected messages. I saved those predictions in `data/predictions`, so everyone starts from the same numbers. `p_yes` is the model's probability that a message needs follow-up.

**Which set plays which role?** The 23 customer messages are too few for three parts, so they have no test set:

| Customer messages (Module 2 names) | Role in Modules 8–9 | CFPB course data (Part 2) |
|---|---|---|
| 13 development messages | training: fit the model | `train` (6,000) |
| 10 protected messages | validation: compare and choose | `valid` (2,000) |
| none | test: check once at the end | `test` (2,000), sealed until 2.10 |

So in Part 1 you choose a threshold on the protected messages. That is the validation role. The "choose on validation, test once" rule is practised in Part 2.

In [ ]:
messages = pd.read_csv(DATA / "predictions/m08_feedback_predictions.csv")
# y = 1 for "yes" and 0 for "no": astype(int) turns True/False into 1/0.
messages["y"] = (messages["follow_up"] == "yes").astype(int)
# Sort from the highest probability down, so the table reads like a ranked queue.
messages = messages.sort_values("p_yes", ascending=False).reset_index(drop=True)
messages[["feedback_id", "follow_up", "p_yes", "text"]]

### 1.4 A threshold turns a probability into a decision

A **threshold** is the cut-off: flag a message when `p_yes` is at or above it. Module 8 used 0.5 without saying so. Nothing makes 0.5 special.

The helper below counts, for each threshold, how many messages are flagged and how many of those are right:

- **TP** (true positive): flagged and really needs follow-up.
- **FP** (false positive): flagged but does not need it. Someone's time is wasted.
- **FN** (false negative): not flagged but needs it. A customer is missed.
- **Precision** = TP / flagged. **Recall** = TP / all messages that need follow-up.

In [ ]:
def threshold_table(y, p, thresholds):
    """Take true labels (1/0), probabilities and a list of thresholds; return one row of counts per threshold."""
    rows = []
    for t in thresholds:
        # True for every message we flag at this threshold.
        flag = p >= t
        # & means "and"; ~flag means "not flagged". Summing True values counts them.
        tp = int((flag & (y == 1)).sum())
        fp = int((flag & (y == 0)).sum())
        fn = int((~flag & (y == 1)).sum())
        precision = tp / max(tp + fp, 1)  # max(..., 1): no division by zero when nothing is flagged
        recall = tp / (tp + fn)
        rows.append({"threshold": t, "flagged": tp + fp, "TP": tp, "FP": fp, "FN": fn,
                     "precision": round(precision, 3), "recall": round(recall, 3)})
    return pd.DataFrame(rows)

message_table = threshold_table(messages["y"], messages["p_yes"], [0.45, 0.50, 0.55])
message_table

### 1.5 Add a cost table

Errors do not cost the same. The service team estimates (made-up numbers for this exercise):

| Error | Cost |
|---|---:|
| Missed follow-up (FN) | $20 |
| Unnecessary follow-up (FP) | $5 |

The best threshold is the one with the lowest total cost, not the one with the highest accuracy.

In [ ]:
MESSAGE_COST_FN = 20  # dollars per missed follow-up
MESSAGE_COST_FP = 5   # dollars per unnecessary follow-up
# Total cost at each threshold: number of errors of each kind times their cost.
message_table["cost"] = message_table["FN"] * MESSAGE_COST_FN + message_table["FP"] * MESSAGE_COST_FP
message_table

### 1.6 A human-review band

A third option is to **abstain**: the model does not decide, and a person reads the message. We send messages with `p_yes` between 0.47 and 0.53 to review. Above the band the message is flagged automatically; below it, it is not. Assume a review costs $2 (a made-up number) and the reviewer decides correctly.

In [ ]:
MESSAGE_COST_REVIEW = 2  # dollars per human review

# np.select checks the conditions in order: 0.53 or more -> "flag", else 0.47 or more -> "review",
# otherwise "no flag".
messages["action"] = np.select([messages["p_yes"] >= 0.53, messages["p_yes"] >= 0.47], ["flag", "review"], default="no flag")

# Reviewed messages are decided correctly, so only "no flag" and "flag" can be wrong.
fn = ((messages["action"] == "no flag") & (messages["y"] == 1)).sum()
fp = ((messages["action"] == "flag") & (messages["y"] == 0)).sum()
n_reviews = (messages["action"] == "review").sum()
cost = fn * MESSAGE_COST_FN + fp * MESSAGE_COST_FP + n_reviews * MESSAGE_COST_REVIEW
print("reviews:", n_reviews, "| FN:", fn, "| FP:", fp, "| cost:", cost)
messages[["feedback_id", "follow_up", "p_yes", "action"]]

### 1.7 Save your outputs

This writes the threshold table and the actions to `outputs/m09`. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
message_table.to_csv(OUTPUT_DIR / "feedback_thresholds.csv", index=False)
messages[["feedback_id", "follow_up", "p_yes", "action"]].to_csv(OUTPUT_DIR / "feedback_actions.csv", index=False)
print("saved:", OUTPUT_DIR / "feedback_thresholds.csv")
print("saved:", OUTPUT_DIR / "feedback_actions.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. Which of the three thresholds in 1.5 has the lowest cost? Would the answer change if a missed follow-up cost $5 instead of $20?
2. In 1.6, did the review band lower the cost compared with the best single threshold? Why or why not?

**Your answer:**

## Part 2 Practice: an urgent-complaint queue

A complaint team wants a fast queue for **urgent** complaints: fraud, scams, transactions the consumer did not authorize, and identity theft. These need action within a day, before more money is lost.

**The target.** We define urgent with a written rule on the issue and sub-issue the consumer chose. These fields are used **only to make the label**, never as features. Module 8 showed that they leak. The model sees the narrative only.

### 2.1 Load the complaints and make the urgent label

In [ ]:
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")

# The written rule: a complaint is urgent if its issue or its sub-issue is in one of these sets.
URGENT_ISSUES = {"Fraud or scam",
                 "Unauthorized transactions or other transaction problem",
                 "Problem with fraud alerts or security freezes"}
URGENT_SUB_ISSUES = {"Transaction was not authorized",
                     "Card was charged for something you did not purchase with the card",
                     "Account opened without my consent or knowledge",
                     "Card opened without my consent or knowledge",
                     "Debt was result of identity theft"}
urgent_issue = complaints["issue"].isin(URGENT_ISSUES)
urgent_sub_issue = complaints["sub_issue"].isin(URGENT_SUB_ISSUES)
# | means "or"; astype(int) turns True/False into 1/0.
complaints["urgent"] = (urgent_issue | urgent_sub_issue).astype(int)

# Look only at train and valid: the test split stays sealed until 2.10.
open_splits = complaints[complaints["split"].isin(["train", "valid"])]

# Per split: number of urgent complaints (sum of the 1s), all complaints (size) and the share (mean).
by_split = open_splits.groupby("split")["urgent"]
split_summary = by_split.agg(urgent="sum", n="size", share="mean")
split_summary.round(3)

### 2.2 Where are the urgent complaints?

The rule depends on the menus each product offers. Mortgage has no fraud option on its menu, so no mortgage complaint is urgent by this rule. That is a limit of the label, not a fact about mortgages.

The video counted all 10,000 complaints. Here we count only `train` and `valid`, so the test split stays sealed. You see the test share in 2.10.

In [ ]:
# Urgent complaints and all complaints per product (train and valid only), most urgent first.
by_product = open_splits.groupby("product")["urgent"].agg(urgent="sum", n="size")
by_product.sort_values("urgent", ascending=False)

### 2.3 Fit the urgent model and score the validation split

Same Pipeline and settings as Module 8: TF-IDF on the narrative, then logistic regression. It is fitted on `train` only.

**Average precision (AP)** summarizes the precision-recall trade-off in one number between 0 and 1. A model that ranks at random gets about the share of urgent complaints, so compare AP with that share.

In [ ]:
train = complaints[complaints["split"].eq("train")]
valid = complaints[complaints["split"].eq("valid")].copy()

def urgent_model():
    """Return a new, unfitted pipeline with the Module 8 settings."""
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words=["xx", "xxxx"]),
                         LogisticRegression(max_iter=1000))

# Fit on train only.
model = urgent_model()
model.fit(train["narrative"], train["urgent"])

# Column 1 of predict_proba is the probability of class 1 (urgent).
valid["p_urgent"] = model.predict_proba(valid["narrative"])[:, 1]
print("urgent share in valid:", round(valid["urgent"].mean(), 3))
print("average precision on valid:", round(average_precision_score(valid["urgent"], valid["p_urgent"]), 3))

### 2.4 Row split or time split?

Our splits were made by drawing complaints at random (a **row split**), after keeping copies of the same template letter together (a **grouped split**). But the queue will run on **future** complaints. A **time split** asks the deployment question: train on the past, check on later months.

We pool `train` and `valid` (the test split stays sealed). The time split trains on July–October and checks on November–December. The row split checks on a random sample of the same size. This row split redraws single complaints at random; it does not keep copies of a template letter together. Each fit takes a few seconds.

AP depends on how common urgent complaints are, so the table also shows AP divided by the urgent share. This is AP relative to the approximate random-ranking baseline. It does not mean recall or business value is multiplied by that amount. On its own, the ratio cannot separate the effect of prevalence from a change over time or from template overlap.

In [ ]:
# Pool train and valid; the test split stays sealed.
pool = complaints[complaints["split"].isin(["train", "valid"])].reset_index(drop=True)

# Time split: evaluate on complaints received from November 1, 2025 (dates compare as text here).
late = pool["date_received"] >= "2025-11-01"

# Row split: evaluate on a random sample of the same size. random_state=6403 fixes the draw for everyone.
random_sample = pool.sample(n=late.sum(), random_state=6403)
random_eval = pool.index.isin(random_sample.index)

results = []
# .to_numpy() turns the pandas True/False column into a plain array, like random_eval.
for name, is_eval in [("row split", random_eval), ("time split", late.to_numpy())]:
    # Fit on the rows not used for evaluation (~ means "not"), then score the evaluation rows.
    split_model = urgent_model()
    split_model.fit(pool.loc[~is_eval, "narrative"], pool.loc[~is_eval, "urgent"])
    p_eval = split_model.predict_proba(pool.loc[is_eval, "narrative"])[:, 1]
    y_eval = pool.loc[is_eval, "urgent"]
    results.append({
        "split": name,
        "train": int((~is_eval).sum()),
        "evaluate": int(is_eval.sum()),
        "urgent_share": round(y_eval.mean(), 3),
        "average_precision": round(average_precision_score(y_eval, p_eval), 3),
    })
split_table = pd.DataFrame(results)
split_table["AP_to_share"] = (split_table["average_precision"] / split_table["urgent_share"]).round(1)
split_table

### 2.5 The precision-recall curve

Each point on the curve is one threshold. Moving left to right, the threshold falls and the model flags more complaints. Recall cannot decrease. Precision often falls overall, but it can rise at individual points. The dots mark three thresholds we inspect below.

In [ ]:
# One (precision, recall) point for every possible threshold.
precision, recall, thresholds = precision_recall_curve(valid["urgent"], valid["p_urgent"])
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(recall, precision)

# Mark the three thresholds on the curve.
for t in [0.10, 0.20, 0.30]:
    # np.abs(thresholds - t) is the distance of every threshold from t;
    # np.argmin gives the position of the smallest distance, so the closest threshold.
    distance = np.abs(thresholds - t)
    i = np.argmin(distance)
    ax.plot(recall[i], precision[i], "o")
    ax.annotate(f"{t:.2f}", (recall[i], precision[i]), textcoords="offset points", xytext=(5, 5))
# The dashed line is the urgent share: the precision of a random ranking.
ax.axhline(valid["urgent"].mean(), linestyle="--", color="gray")
ax.set_xlabel("recall (share of urgent complaints found)")
ax.set_ylabel("precision (share of flags that are urgent)")
ax.set_title("Urgent model on the validation split")
plt.show()

### 2.6 Three thresholds and a cost table

The complaint team supplied this cost table (made-up numbers for this exercise, per complaint):

| Outcome | Cost |
|---|---:|
| Missed urgent complaint (FN): more money lost, a regulator follow-up | $100 |
| Non-urgent complaint in the urgent queue (FP): specialist time | $10 |
| Human triage of one complaint (review band) | $3 |

Compare three thresholds on the 2,000 validation complaints.

In [ ]:
COST_FN = 100     # dollars per missed urgent complaint
COST_FP = 10      # dollars per non-urgent complaint in the urgent queue
COST_REVIEW = 3   # dollars per human triage

# The helper from 1.4, now on the 2,000 validation complaints.
valid_table = threshold_table(valid["urgent"], valid["p_urgent"], [0.10, 0.20, 0.30])
valid_table["cost"] = valid_table["FN"] * COST_FN + valid_table["FP"] * COST_FP
valid_table

### 2.7 A human-review band

Now there are three actions:

- 0.30 or more: straight to the urgent queue.
- From 0.10 up to 0.30: a person triages the complaint.
- Below 0.10: the normal queue.

We assume the person sorts the reviewed complaints correctly. The helper returns the counts and the cost. You use it again for the frozen policy.

The two thresholds must be in order: `0 <= review_from <= auto_urgent <= 1`. Equal thresholds mean no review band.

In [ ]:
def apply_policy(y, p, review_from, auto_urgent):
    """Take true labels, probabilities and the two thresholds; return the counts, recall and cost."""
    # Each complaint gets one of three actions.
    auto = p >= auto_urgent                  # urgent queue
    review = (p >= review_from) & ~auto      # human triage
    normal = ~(auto | review)                # normal queue: everything else
    # Urgent complaints in the normal queue are missed (FN); non-urgent ones in the urgent queue are FP.
    # Triage sorts its complaints correctly, so it adds no errors, only its cost.
    fn = int((normal & (y == 1)).sum())
    fp = int((auto & (y == 0)).sum())
    n_urgent = int((y == 1).sum())
    found = n_urgent - fn
    n_reviews = int(review.sum())
    return {"auto_urgent": int(auto.sum()), "reviews": n_reviews, "FN": fn, "FP": fp,
            "urgent_found": found, "recall": round(found / n_urgent, 3),
            "cost": fn * COST_FN + fp * COST_FP + n_reviews * COST_REVIEW}

# Your policy from Exercise 3. It stays empty until you write it there.
my_policy = None
READY_TO_FREEZE = False

band = apply_policy(valid["urgent"], valid["p_urgent"], review_from=0.10, auto_urgent=0.30)
band

### 2.8 Recurring mistakes

Ten validation errors were picked for you: five urgent complaints with a low probability (missed) and five non-urgent complaints with a high probability (false alarms). The table shows the issue the consumer chose. Below it, the cell prints each full narrative, because a detail that decides the category can come late in the text.

In [ ]:
ERROR_IDS = [14980183, 14650359, 17194527, 14949635, 14905757,
             14876285, 14874590, 14592566, 16372496, 16032208]
# Look up the ten complaints by ID, in the order of ERROR_IDS.
errors = valid.set_index("complaint_id").loc[ERROR_IDS, ["urgent", "p_urgent", "issue", "sub_issue", "narrative"]]
errors["p_urgent"] = errors["p_urgent"].round(3)
print(errors.drop(columns="narrative").to_string())

# Print each full narrative. iterrows goes through the table one row at a time.
for complaint_id, row in errors.iterrows():
    print()
    print(f"{complaint_id} | urgent {row['urgent']} | p_urgent {row['p_urgent']}")
    print(row["narrative"])

### Exercise 1: choose one threshold under a capacity limit

The urgent team can handle at most **250** flagged complaints per 2,000 (no review band). Use `threshold_table` with thresholds from 0.05 to 0.50 in steps of 0.05 (`np.arange(0.05, 0.51, 0.05).round(2)`), add the cost column, and keep the rows with at most 250 flagged. Which threshold do you choose? In a comment, give its cost, its recall, and one sentence on why.

In [ ]:
# Exercise 1: your code here
# TODO: make the table with threshold_table(valid["urgent"], valid["p_urgent"], np.arange(0.05, 0.51, 0.05).round(2)).
# TODO: add the cost column as in 2.6: FN * COST_FN + FP * COST_FP.
# TODO: keep the rows with at most 250 flagged and find the one with the lowest cost.
# TODO: in a comment, give the chosen threshold's cost and recall, and one sentence on why.

### Exercise 2: an error taxonomy

Put each of the ten complaints from 2.8 into one category:

- **A. Template letter:** the label says identity theft or fraud, but the text is a general dispute letter with no fraud story.
- **B. Urgent text, other menu choice:** the text describes fraud, a scam or unauthorized use, but the consumer picked a non-urgent issue.
- **C. Missed urgent text:** the text clearly tells an urgent story and the label agrees, but the model scored it low.
- **D. False alarm:** the text is not about fraud; the model reacted to related words.
- **Unclear:** the full text does not let you decide. Use `"unclear"` for these.

Make a dictionary `taxonomy = {complaint_id: "A", ...}`, print the count per category, and name **one** improvement that the counts justify. These ten were picked by hand, so describe what you see "among these ten" complaints, not across all errors.

In [ ]:
# Exercise 2: your code here
# TODO: read the ten complaints from 2.8 and write taxonomy = {complaint_id: "A", ...}, one letter each.
# TODO: print the number of complaints per category (for example with pd.Series(taxonomy).value_counts()).
# TODO: in a comment, name one improvement that the counts justify among these ten complaints.

### Exercise 3: write your policy

Write the policy you recommend as a dictionary named `my_policy` with two thresholds, for example `{"review_from": 0.10, "auto_urgent": 0.30}`. Choose `0 <= review_from <= auto_urgent <= 1`; set them equal if you want no review band. Use only what you learned on the validation split. Print its validation result with `apply_policy`, and add a comment with your reason.

The capacity limit from Exercise 1 still holds: at most **250** complaints per 2,000 may go straight to the urgent queue (`auto_urgent` in the result). Triage is done by a separate team, so the review band does not count against that limit.

When you are sure, add the line `READY_TO_FREEZE = True` at the end of your cell. The next cell freezes your policy only then.

In [ ]:
# Exercise 3: your code here
# TODO: write my_policy = {"review_from": ..., "auto_urgent": ...} using only the validation results.
# TODO: print apply_policy(valid["urgent"], valid["p_urgent"], my_policy["review_from"], my_policy["auto_urgent"]).
# TODO: check that "auto_urgent" in the result is at most 250, and add a comment with your reason.
# TODO: when you are sure, add the line READY_TO_FREEZE = True at the end of this cell.

### 2.9 Freeze the policy

**Freezing** means writing the policy down, with a time stamp, **before** you look at the test split. If you change the thresholds after seeing the test result, the test result no longer tells you how the policy will do on new complaints.

The cell writes `outputs/m09/frozen_policy.json` only once. If the file already exists, it loads the policy from the file and does not change it. If you have not finished Exercise 3, it freezes nothing and the test split stays sealed.

In [ ]:
policy_file = OUTPUT_DIR / "frozen_policy.json"

if policy_file.exists():
    # Frozen before: load it and leave it unchanged.
    frozen = json.loads(policy_file.read_text())
    print("Policy already frozen at", frozen["frozen_at"], "- it is not changed.")
# my_policy and READY_TO_FREEZE start empty in 2.7; Exercise 3 sets them.
elif my_policy is not None and READY_TO_FREEZE:
    # Write down the thresholds, the model, the costs and the validation result, with a time stamp.
    frozen = dict(my_policy)  # a copy of your two thresholds
    frozen["model"] = "TF-IDF (1-2 words, min_df=5) + logistic regression, fitted on train"
    frozen["costs"] = {"FN": COST_FN, "FP": COST_FP, "review": COST_REVIEW}
    frozen["costs_note"] = "made-up costs for this exercise, supplied by the course"
    frozen["validation_result"] = apply_policy(valid["urgent"], valid["p_urgent"],
                                               my_policy["review_from"], my_policy["auto_urgent"])
    frozen["frozen_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")  # time now, in UTC
    policy_file.write_text(json.dumps(frozen, indent=1))
    print("Policy frozen now.")
else:
    frozen = None
    print("Nothing frozen yet. Finish Exercise 3, set READY_TO_FREEZE = True, then run this cell again.")

# Show the frozen policy.
if frozen is not None:
    print(json.dumps(frozen, indent=1))

### 2.10 Open the test split once: the urgent policy

Now apply the frozen policy to the 2,000 test complaints. The model is the same one, fitted on `train`. Do not go back to change the policy after this step.

The first run scores the test split and saves the result, with its time stamp, in `outputs/m09/test_result.json`. Later runs only display that saved result. So Run All reuses your recorded test result and does not open the test split again.

In [ ]:
result_file = OUTPUT_DIR / "test_result.json"

if frozen is None:
    print("The test split stays sealed: freeze your policy in 2.9 first.")
elif result_file.exists():
    # Opened before: show the saved result and do not score the test split again.
    test_record = json.loads(result_file.read_text())
    print("Test split opened at", test_record["opened_at"], "- showing the saved result.")
else:
    # Score the test complaints with the model fitted in 2.3 and apply the frozen thresholds.
    test = complaints[complaints["split"].eq("test")].copy()
    test["p_urgent"] = model.predict_proba(test["narrative"])[:, 1]
    test_record = {
        "opened_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),  # time now, in UTC
        "urgent_share": round(test["urgent"].mean(), 3),
        "average_precision": round(average_precision_score(test["urgent"], test["p_urgent"]), 3),
        "result": apply_policy(test["urgent"], test["p_urgent"], frozen["review_from"], frozen["auto_urgent"]),
    }
    result_file.write_text(json.dumps(test_record, indent=1))
    print("Test result saved in", result_file)

if frozen is not None:
    print("urgent share in test:", test_record["urgent_share"], "| average precision on test:",
          test_record["average_precision"])
    # Validation and test side by side; :>13 and :>6 right-align the values so they line up.
    for key, value in test_record["result"].items():
        valid_value = frozen["validation_result"][key]
        print(f"{key:>13}: validation {valid_value:>6} | test {value:>6}")

### 2.11 Open the test split once: the Module 8 routers

Module 8 sealed its test predictions. Here we score them once, from the saved file in `data/predictions`. Saved LLM labels exist for 300 test complaints, so all methods are also compared on those 300. This cell also waits until your policy is frozen.

In [ ]:
if frozen is None:
    print("The test split stays sealed: freeze your policy in 2.9 first.")
else:
    routes = pd.read_csv(DATA / "predictions/m08_route_predictions.csv")
    rows = []
    for split in ["valid", "test"]:
        # All complaints of this split, and the 300 with saved LLM labels.
        part = routes[routes["split"].eq(split)]
        sub = part[part["in_llm_subset"]]
        rows.append({
            "split": split,
            "TF-IDF (all)": f1_score(part["route"], part["tfidf_route"], average="macro"),
            "embedding (all)": f1_score(part["route"], part["emb_route"], average="macro"),
            "TF-IDF (300)": f1_score(sub["route"], sub["tfidf_route"], average="macro"),
            "embedding (300)": f1_score(sub["route"], sub["emb_route"], average="macro"),
            "LLM (300)": f1_score(sub["route"], sub["llm_route"], average="macro"),
        })
    print(pd.DataFrame(rows).set_index("split").round(3).to_string())

### Practice interpretation

Write 3–5 sentences:

- Using 2.4, how did the time split differ from the row split (AP and AP divided by the urgent share), and which one matches how the queue will be used?
- Using 2.6, 2.7 and Exercise 1, why is 0.5 a poor default threshold for this model, and which policy did you freeze? Give its validation cost, recall and number of complaints sent straight to the urgent queue.
- Using 2.10 and Exercise 2, did the frozen policy hold up on the test split? Which error category was most common, and what one improvement would you make?

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Run All reuses your recorded test result from 2.10, so the test split is not opened again. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Calibration: does 0.25 mean 25 percent?

A model is **calibrated** when its probabilities match what happens: of the complaints scored about 0.25, about 25% are urgent. A **reliability table** checks this. It puts the validation complaints into probability bins and compares the average predicted probability with the observed share of urgent complaints. Always look at the counts: a bin with 5 complaints says little.

In [ ]:
# np.linspace(0, 1, 11) gives 11 edges, so 10 bins of width 0.1; pd.cut puts each complaint in one.
bins = pd.cut(valid["p_urgent"], bins=np.linspace(0, 1, 11), include_lowest=True)
# Per bin: number of complaints, mean predicted probability and observed share of urgent complaints.
# observed=True leaves out empty bins.
reliability = valid.groupby(bins, observed=True).agg(complaints=("urgent", "size"),
                                                     mean_predicted=("p_urgent", "mean"),
                                                     observed_urgent=("urgent", "mean")).round(3)
reliability

**Calibration training.** `CalibratedClassifierCV` refits the model in five folds of `train` and learns a mapping from scores to probabilities. Compare its reliability table with the one above. This takes about half a minute. Note that new probabilities would need new thresholds, chosen again on validation.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
# cv=5: five folds of train; method="isotonic": a flexible step-shaped mapping.
calibrated = CalibratedClassifierCV(urgent_model(), method="isotonic", cv=5)
calibrated.fit(train["narrative"], train["urgent"])
valid["p_calibrated"] = calibrated.predict_proba(valid["narrative"])[:, 1]

# The same reliability table for the calibrated probabilities.
calibrated_bins = pd.cut(valid["p_calibrated"], bins=np.linspace(0, 1, 11), include_lowest=True)
valid.groupby(calibrated_bins, observed=True).agg(complaints=("urgent", "size"),
                                                  mean_predicted=("p_calibrated", "mean"),
                                                  observed_urgent=("urgent", "mean")).round(3)

### B. How much does one random row split vary?

Repeat the row split of 2.4 with three other random seeds. How wide is the range, compared with the gap to the time split?

In [ ]:
for seed in [1, 2, 3]:
    # The same row split as 2.4, with a different random draw.
    random_sample = pool.sample(n=late.sum(), random_state=seed)
    is_eval = pool.index.isin(random_sample.index)
    split_model = urgent_model()
    split_model.fit(pool.loc[~is_eval, "narrative"], pool.loc[~is_eval, "urgent"])
    p_eval = split_model.predict_proba(pool.loc[is_eval, "narrative"])[:, 1]
    print("seed", seed, "row split AP:", round(average_precision_score(pool.loc[is_eval, "urgent"], p_eval), 3))

### C. SMOTE and class weights

SMOTE creates synthetic minority examples. It needs the `imbalanced-learn` package (not in the course environment), and on sparse TF-IDF features its synthetic "complaints" are hard to interpret. A simpler option is `class_weight="balanced"` in logistic regression. Try it and compare AP and the reliability table. Remember that it shifts every probability, so thresholds must be chosen again.

In [ ]:
# Same pipeline, but class_weight="balanced" gives the rare urgent class more weight.
weighted = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words=["xx", "xxxx"]),
                         LogisticRegression(max_iter=1000, class_weight="balanced"))
weighted.fit(train["narrative"], train["urgent"])
p_weighted = weighted.predict_proba(valid["narrative"])[:, 1]
print("balanced class weights: AP", round(average_precision_score(valid["urgent"], p_weighted), 3),
      "| mean probability", round(p_weighted.mean(), 3), "| urgent share", round(valid["urgent"].mean(), 3))

## Further reading

- [scikit-learn user guide: tuning the decision threshold](https://scikit-learn.org/stable/modules/classification_threshold.html)
- [scikit-learn user guide: probability calibration](https://scikit-learn.org/stable/modules/calibration.html)
- [scikit-learn example: precision-recall](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html)
- [Kapoor and Narayanan (2023), Leakage and the reproducibility crisis in machine-learning-based science](https://doi.org/10.1016/j.patter.2023.100804)

Next, Module 10 discovers themes in complaints when no labels exist yet.